In [1]:
import math
import time
import numpy as np
import pandas as pd
import yfinance as yf
import os
import sys


import torch
import torch.nn as nn


REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../.."))
sys.path.insert(0, REPO_ROOT)
from model import train_pinn


from montecarlo import mc_asian_call_arith
from montecarlo import pinn_price_real
from montecarlo_PIDE import mc_asian_call_arith_jump

from archive.evaluation import evaluate_mae_pinn_vs_mc
from archive.evaluation import parity_plot_plotly
from archive.evaluation import run_full_evaluation
from evaluation_PIDE import run_full_evaluation_PIDE

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float32


In [2]:
# Get real WTI data and infer ranges

df = yf.download("CL=F", start="2010-01-01", progress=False)
prices = df["Close"].dropna()


S0 = float(prices.iloc[-1])
S_max_hist = float(prices.max())


log_ret = np.log(prices / prices.shift(1)).dropna()
sigma_hist = float(log_ret.std()) * np.sqrt(252)


T = 1.0
S_max_real = 1.2 * S_max_hist
I_max_real = S_max_real * T
sigma_max = min(1.0, 2.0 * sigma_hist)
r_max = 0.5


print("=== REAL (USD) ranges ===")
print(f"S0        = {S0:.2f}")
print(f"S_max     = {S_max_real:.2f}")
print(f"I_max     = {I_max_real:.2f}")
print(f"sigma     = {sigma_hist:.3f}")
print(f"sigma_max = {sigma_max:.3f}")
print(f"r_max     = {r_max}")


YF.download() has changed argument auto_adjust default to True
=== REAL (USD) ranges ===
S0        = 57.32
S_max     = 148.44
I_max     = 148.44
sigma     = 0.404
sigma_max = 0.807
r_max     = 0.5


/var/folders/7y/h7b4hxf568n1y6xlpl8vf6340000gn/T/ipykernel_89877/968340630.py:7: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  S0 = float(prices.iloc[-1])
/var/folders/7y/h7b4hxf568n1y6xlpl8vf6340000gn/T/ipykernel_89877/968340630.py:8: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  S_max_hist = float(prices.max())
/opt/homebrew/lib/python3.11/site-packages/pandas/core/internals/blocks.py:395: RuntimeWarning: invalid value encountered in log
  result = func(self.values, **kwargs)
/var/folders/7y/h7b4hxf568n1y6xlpl8vf6340000gn/T/ipykernel_89877/968340630.py:12: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  sigma_hist = float(log_ret.std()) * np.sqrt(252)


In [3]:
# Normalization: K_scale = S0  (so S̃0 = 1)

K_scale = S0  


S_max = S_max_real / K_scale
I_max = I_max_real / K_scale


print("\n=== NORMALIZED (dimensionless) ranges ===")
print(f"K_scale   = {K_scale:.2f}")
print(f"S0_tilde  = {S0 / K_scale:.3f}")      # should be 1.000
print(f"S_max_t   = {S_max:.3f}")
print(f"I_max_t   = {I_max:.3f}")
print(f"r_max     = {r_max:.3f}")
print(f"sigma_max = {sigma_max:.3f}")



=== NORMALIZED (dimensionless) ranges ===
K_scale   = 57.32
S0_tilde  = 1.000
S_max_t   = 2.590
I_max_t   = 2.590
r_max     = 0.500
sigma_max = 0.807


In [8]:
model = train_pinn(
   S_max, I_max, r_max, sigma_max,
   width=160,
   depth=4,
   n_epochs=20000,     # start here; increase toward 200k/500k if needed
   lr0=1e-3,
   Np=1000,
   n_bc_axis=100,
   w_pde=0.6,
   print_every=2000
)

K_real = S0  # ATM
r_eval = 0.05
sigma_eval = sigma_hist



print("Evaluation (real scale)")
test_S = [0.8*S0, 1.0*S0, 1.2*S0]  # around spot


for S_test in test_S:
   pinn_p = pinn_price_real(model, S_test, K_real, r_eval, sigma_eval, t0=0.0)
   mc_p, mc_se = mc_asian_call_arith(S_test, K_real, r_eval, sigma_eval, T=1.0, n_steps=252, n_paths=50_000, antithetic=True)


   print(f"S={S_test:8.2f} | PINN={pinn_p:10.4f} | MC={mc_p:10.4f} ± {1.645*mc_se:8.4f} (90% CI half-width)")


ep=   2000 | loss=5.335e-04 | pde=3.514e-04 | data=3.227e-04 | lr=9.76e-04 | 3.6 min
ep=   4000 | loss=2.182e-04 | pde=1.249e-04 | data=1.433e-04 | lr=9.05e-04 | 6.8 min
ep=   6000 | loss=1.459e-04 | pde=7.867e-05 | data=9.872e-05 | lr=7.94e-04 | 10.1 min
ep=   8000 | loss=2.852e-04 | pde=4.582e-05 | data=2.577e-04 | lr=6.55e-04 | 13.0 min
ep=  10000 | loss=5.723e-05 | pde=3.285e-05 | data=3.752e-05 | lr=5.00e-04 | 15.5 min
ep=  12000 | loss=2.769e-05 | pde=2.146e-05 | data=1.482e-05 | lr=3.45e-04 | 18.0 min
ep=  14000 | loss=2.195e-05 | pde=1.861e-05 | data=1.078e-05 | lr=2.06e-04 | 20.5 min
ep=  16000 | loss=1.931e-05 | pde=1.336e-05 | data=1.129e-05 | lr=9.55e-05 | 23.1 min
ep=  18000 | loss=2.334e-05 | pde=1.321e-05 | data=1.541e-05 | lr=2.45e-05 | 25.7 min
ep=  20000 | loss=1.481e-05 | pde=1.289e-05 | data=7.081e-06 | lr=0.00e+00 | 28.2 min
Evaluation (real scale)
S=   60.00 | PINN=    2.5888 | MC=    1.7932 ±   0.0424 (90% CI half-width)
S=   75.00 | PINN=    8.9547 | MC=    7.74

In [10]:
S_grid_eval = np.linspace(0, S_max_real, 10)

results = run_full_evaluation_PIDE(
    model=model,          # PINN già addestrata
    S_grid=S_grid_eval,   # grid su S
    K=K_real,        # strike reale
    r=r_eval,             # risk-free rate
    sigma=sigma_eval,     # volatilità
    T=1.0,                # maturità
    n_steps=252,       # discretizzazione MC
    n_paths=200_000,   # paths MC
    h_delta=1.0,          # bump per delta
)

MAE (PINN vs MC): 0.071801
Speed-up (MC / PINN): 30021.6x


In [6]:
import torch
import numpy as np
import math

# ============================================================
# Setup
# ============================================================

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float32

# ============================================================
# Monte Carlo Asian Call – Merton Jump–Diffusion
# ============================================================

def mc_asian_call_arith_jump(
    S0_real, K_real, r, sigma,
    lambda_jump, mu_J, sigma_J,
    T=1.0,
    n_steps=252,
    n_paths=200_000,
    antithetic=True
):
    dt = T / n_steps
    disc = math.exp(-r * T)

    # Brownian part
    if antithetic:
        half = n_paths // 2
        Z = np.random.normal(size=(half, n_steps))
        Z = np.vstack([Z, -Z])
    else:
        Z = np.random.normal(size=(n_paths, n_steps))

    n_sim = Z.shape[0]

    # Poisson jumps
    N_jump = np.random.poisson(lambda_jump * dt, size=(n_sim, n_steps))

    S = np.full((n_sim,), S0_real, dtype=float)
    sumS = np.zeros_like(S)

    # Drift compensation
    kappa = math.exp(mu_J + 0.5 * sigma_J**2) - 1.0
    drift = (r - lambda_jump * kappa - 0.5 * sigma**2) * dt
    vol = sigma * math.sqrt(dt)

    for k in range(n_steps):
        # diffusion
        S *= np.exp(drift + vol * Z[:, k])

        # jumps
        jump_mask = N_jump[:, k] > 0
        if np.any(jump_mask):
            Y = np.random.normal(mu_J, sigma_J, size=jump_mask.sum())
            S[jump_mask] *= np.exp(Y)

        sumS += S

    A = sumS / n_steps
    payoff = np.maximum(A - K_real, 0.0)

    price = disc * payoff.mean()
    stderr = disc * payoff.std(ddof=1) / math.sqrt(len(payoff))

    return price, stderr


# ============================================================
# PINN pricing (real scale, K-normalized training)
# ============================================================

@torch.no_grad()
def pinn_price_real(model, S0_real, K_real, r, sigma, t0=0.0):
    S_tilde = torch.tensor([[S0_real / K_real]], device=DEVICE, dtype=DTYPE)
    I_tilde = torch.tensor([[0.0]], device=DEVICE, dtype=DTYPE)
    t = torch.tensor([[t0]], device=DEVICE, dtype=DTYPE)
    rr = torch.tensor([[r]], device=DEVICE, dtype=DTYPE)
    ss = torch.tensor([[sigma]], device=DEVICE, dtype=DTYPE)

    X = torch.cat([S_tilde, I_tilde, t, rr, ss], dim=1)
    V_tilde = model(X).item()

    return K_real * V_tilde


# ============================================================
# Global evaluation (NO region filtering)
# ============================================================

def evaluate_pinn_vs_mc_jump(
    model,
    S_grid,
    K, r, sigma, T,
    lambda_jump, mu_J, sigma_J,
    n_paths_mc=200_000
):
    pinn_prices = []
    mc_prices   = []

    for S in S_grid:
        p_pinn = pinn_price_real(model, S, K, r, sigma)

        p_mc, _ = mc_asian_call_arith_jump(
            S, K, r, sigma,
            lambda_jump, mu_J, sigma_J,
            T=T,
            n_paths=n_paths_mc
        )

        pinn_prices.append(p_pinn)
        mc_prices.append(p_mc)

    pinn_prices = np.array(pinn_prices)
    mc_prices   = np.array(mc_prices)

    mae  = np.mean(np.abs(pinn_prices - mc_prices))
    rmse = np.sqrt(np.mean((pinn_prices - mc_prices)**2))

    return {
        "S_grid": S_grid,
        "PINN": pinn_prices,
        "MC": mc_prices,
        "MAE": mae,
        "RMSE": rmse
    }


In [7]:

S0 = 75.0
K  = 75.0
r  = 0.03
sigma = 0.32
T = 1.0

lambda_jump = 2.188
mu_J        = 0.0196
sigma_J     = 0.1817

# Griglia standard
S_grid_eval = np.linspace(0, S_max_real, 100)

results = evaluate_pinn_vs_mc_jump(
    model,
    S_grid_eval,
    K, r, sigma, T,
    lambda_jump, mu_J, sigma_J
)

print("MAE :", results["MAE"])
print("RMSE:", results["RMSE"])


MAE : 0.4894741790579943
RMSE: 0.593401742509329
